# Integração Okta Gateway Auth

AgentCore Identity permite validar acesso de entrada (Inbound Auth) para usuários e aplicações chamando agentes ou ferramentas em um AgentCore Runtime ou validar acesso a alvos do AgentCore Gateway. Ele também fornece acesso de saída seguro (Outbound Auth) de um agente para serviços externos ou um alvo Gateway. Ele se integra com seus provedores de identidade existentes (como Amazon Cognito) enquanto impõe limites de permissão para agentes agindo independentemente ou em nome de usuários (via OAuth).

Inbound Auth valida chamadores tentando invocar agentes ou ferramentas, seja hospedados no AgentCore Runtime, AgentCore Gateway ou em outros ambientes. Inbound Auth funciona com IAM (credenciais SigV4) ou com autorização OAuth.

Por padrão, Amazon Bedrock AgentCore usa credenciais IAM, significando que requisições de usuário ao agente são autenticadas com as credenciais IAM do usuário. Neste tutorial usaremos OAuth com um IDP Okta, então você precisará especificar o seguinte ao configurar seus recursos AgentCore Runtime ou endpoints AgentCore Gateway:

- URL do servidor de descoberta OAuth — Uma string que deve corresponder ao padrão ^.+/.well-known/openid-configuration$ para URLs de descoberta OpenID Connect

- Audiências permitidas — Lista de audiências permitidas para token JWT

- Clientes permitidos — Lista de identificadores de clientes permitidos

Se você usar a CLI AgentCore, pode especificar o tipo de autorização (e servidor de descoberta OAuth) para um AgentCore Runtime quando usar o comando configure. Você também pode usar a operação CreateAgentRuntime e o console Amazon Bedrock AgentCore. Se estiver criando um Gateway, você usa a operação CreateGateway, ou o console.

Antes que o usuário possa usar o agente, a aplicação cliente deve fazer o usuário se autenticar com o autorizador OAuth. Seu cliente recebe um token bearer que então passa para o agente em uma requisição de invocação. Ao receber, o agente valida o token com o servidor de autorização antes de permitir acesso.

## Visão Geral

Neste tutorial configuraremos Inbound Auth usando Okta como provedor de Identidade. Você configurará um tenant Okta com um usuário e um cliente de aplicação. Você aprenderá como hospedar seu agente existente, usando Amazon Bedrock AgentCore Runtime com Inbound Auth usando o cliente de aplicação Okta. Você também configurará um Amazon Bedrock AgentCore Gateway que usa Okta para Inbound Auth com o qual seu agente irá interagir.

### Arquitetura do Tutorial

<figure>
    <img src="images/16.png">
</figure>

### Detalhes do Tutorial

| Informação         | Detalhes                                                                         |
|:-------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial   | Conversacional                                                                   |
| Tipo de agente     | Único                                                                            |
| Framework agêntico | Strands Agents                                                                   |
| Modelo LLM         | Anthropic Claude Sonnet 4                                                        |
| Componentes        | Hospedar agente no AgentCore Runtime. Usando Strands Agent e Amazon Bedrock Model |
| Vertical           | Cross-vertical                                                                   |
| Complexidade       | Fácil                                                                            |
| Inbound Auth       | Okta                                                                             |
| SDK usado          | Amazon BedrockAgentCore Python SDK e boto3                                       |


### Recursos Principais

* Hospedar Agentes no Amazon Bedrock AgentCore Runtime com Inbound e Outbound Auth usando Okta
* Hospedar um Amazon Bedrock AgentCore Gateway com Inbound Auth usando Okta
* Usar modelos Amazon Bedrock
* Usar Strands Agents

## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Direitos IAM para criar novas roles, políticas e usuários IAM
* Direitos IAM para criar um novo AgentCore Agent
* Uma conta Okta
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker rodando

## Configurando o IDP do Okta

Vamos configurar um tenant Okta demo com um cliente de App e um usuário de teste. Usaremos o Okta para fornecer tokens JWT para invocar um agente que implantaremos posteriormente neste tutorial. Se você já tem uma conta Okta, faça login nela.

Navegue até https://developer.okta.com/signup/, selecione "Sign up for Integrator Free Plan" para se cadastrar.

1. Faça login em sua conta.<br><br>
2. Selecione **Directory**, depois **People** e clique em **Add person**.
    <figure>
        <img src="images/9.png">
    </figure><br><br>
3. Preencha o formulário.
    <ol type="1">
        <li>Para <b>Activation</b>, selecione <b>Activate now</b>.</li>
        <li>Marque a caixa <b>I will set password</b> e defina uma senha para o usuário.</li>
        <li>Desmarque a caixa <b>User must change password on first login</b>.</li>
        <li>Clique em <b>Save</b>.</li>
    </ol>

    <figure>
        <img src="images/10.png">
    </figure>
    <br><br>
4. Selecione **Applications**, depois clique em **Create App Integration**.
    <figure>
        <img src="images/1.png">
    </figure>
    <br><br>

5. Para o método de sign-in, selecione **OIDC - OpenID Connect**, depois selecione **Web Application** para o tipo de aplicação e clique em **Next**.

    <figure>
        <img src="images/2.png">
    </figure><br><br>
    <ol type="a">
        <li>Para o nome de integração do App insira <b>Travel Assistant</b>, depois deixe <b>Proof of possession</b> desmarcado e selecione <b>Authorization Code</b> para o tipo de grant.
        <br><br>
        <figure>
            <img src="images/3.png">
        </figure>
        <br><br>
        </li>
        <li>Atualize o URI de sign-in para incluir <b>http://127.0.0.1:5000/callback</b> e <b>https://bedrock-agentcore.us-west-2.amazonaws.com/identities/oauth2/callback</b>. Deixe o URI de redirecionamento de sign-out como está.
        <br><br>
        <figure>
            <img src="images/4.png">
        </figure>
        <br><br>
        </li>
        <li>Em assignments, <b>Allow everyone in your organization to access</b>, depois deixe <b>Enable immediate access</b> marcado. Em seguida, clique em <b>Save</b>.
        <br><br>
        <figure>
            <img src="images/5.png">
        </figure>
        <br><br>
        </li>
        <li>Copie o <b>Client ID</b> e <b>Secret</b> para uso posterior.</li>
        <br><br>
        <figure>
            <img src="images/6.png">
        </figure>
        <br><br>
    </ol><br>

6. No menu do lado esquerdo, selecione **Security**, depois **API**, e clique no nome do seu servidor de autorização.
    <figure>
        <img src="images/17.png">
    </figure><br><br>
    <ol type="a">
        <li>Copie o <b>Audience</b> e salve-o para uso posterior.</li>
            <ul>
                <li><b>Nota</b>: O <b>Audience</b> padrão foi alterado neste exemplo. É recomendado adicionar um novo servidor de autorização se você planeja alterar o audience para que outras aplicações não sejam afetadas.</li><br>
            </ul>
        <figure>
            <img src="images/7.png">
        </figure>
        <br><br>
        <li>Clique em <b>Scopes</b>, depois <b>Add Scope</b>.</li><br>
        <figure>
            <img src="images/43.png">
        </figure>
        <br><br>
        <li>Use <b>okta.myAccount.read</b> para o nome e dê a ele uma <b>Display Phrase</b> e uma <b>Description</b>.</li>
        <li>Defina <b>User Consent</b> como <b>implicit</b>.</li>
        <li>Deixe <b>Block services</b>, <b>Default scope</b> e <b>Metadata</b> nos padrões.</li>
        <li>Clique em <b>Save</b>.
        <figure>
            <img src="images/44.png">
        </figure><br>
        <li>Clique em <b>Claims</b> e adicione as seguintes reivindicações <b>client_id</b> e <b>scope</b>.</li><br>
        <figure>
            <img src="images/8.png">
        </figure><br>
        <li>Clique em <b>Access Policies</b> e depois clique em <b>Add New Access Policy</b></li><br>
        <figure>
            <img src="images/39.png">
        </figure><br>
        <li>Insira um <b>Name</b>, <b>Description</b> e clique em <b>Create Policy</b></li><br>
        <figure>
            <img src="images/40.png">
        </figure><br>
        <li>Clique em <b>Add rule</b></li><br>
        <figure>
            <img src="images/41.png">
        </figure><br>
        <li>Dê à regra um <b>Rule Name</b> e clique em <b>Create Rule</b></li><br>
        <figure>
            <img src="images/42.png">
        </figure><br>
    </ol>




## Criando o API Gateway

Este template CloudFormation cria uma API serverless com autenticação JWT:
* Amazon API Gateway com um autorizador JWT que valida tokens do Okta
* Endpoint protegido em **GET /travel-plans** que requer autenticação JWT válida
* Função AWS Lambda que é executada ao chamar o endpoint **GET /travel-plans**

O AgentCore Gateway transformaria este Amazon API Gateway em um servidor MCP (Model Context Protocol) atuando como um adaptador de protocolo que traduz requisições MCP em chamadas HTTP para os endpoints do API Gateway subjacentes.

Execute a célula de código abaixo para salvar o template CloudFormation como template.yaml no seu sistema de arquivos local.

In [ ]:
%%writefile template.yaml
AWSTemplateFormatVersion: '2010-09-09'
Parameters:
  JwtIssuerUrl:
    Type: String
    Description: The URL of the JWT issuer (e.g., Cognito user pool URL).
    MinLength: 10 # Optional: You can add constraints like minimum length
    MaxLength: 200 # Optional: Maximum length
    # You can also add a Default value if you want

  JwtAudienceList:
    Type: CommaDelimitedList # <-- Using CommaDelimitedList for multiple audiences
    Description: A comma-separated list of expected audience(s) for the JWT (e.g.,
      "my-api-audience-1,my-api-audience-2").
    # Optional: You can add Default or other constraints if needed
    # Default: "my-default-audience"

Resources:
  MyHttpApi:
    Type: AWS::ApiGatewayV2::Api
    Properties:
      Name: MyHttpApi
      ProtocolType: HTTP

  MyJwtAuthorizer:
    Type: AWS::ApiGatewayV2::Authorizer
    Properties:
      ApiId: !Ref MyHttpApi
      AuthorizerType: JWT
      IdentitySource:
        - $request.header.Authorization
      JwtConfiguration:
        Audience: !Ref JwtAudienceList
        Issuer: !Ref JwtIssuerUrl
      Name: MyJwtAuthorizer

  MyLambdaFunction:
    Type: AWS::Lambda::Function
    Properties:
      FunctionName: MyLambdaFunction
      Handler: lambda_function.lambda_handler
      Runtime: python3.12
      Code:
        ZipFile: |
          exports.handler = async (event) => {
              // In non-proxy integration, the Lambda function receives the mapped input
              // from API Gateway, NOT the full HTTP request.
              console.log('Received event:', JSON.stringify(event, null, 2));

              const name = event.name || "World";
              const message = `Hello, ${name}! (from non-proxy integration)`;

              // In non-proxy integration, you can return a simple string, object, etc.
              // API Gateway then formats this into a proper HTTP response using mapping templates.
              return { "message": message }; // Example of a simple JSON response
          };
      MemorySize: 128
      Timeout: 30
      Role: !GetAtt MyLambdaExecutionRole.Arn

  MyLambdaExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service:
                - lambda.amazonaws.com
            Action:
              - sts:AssumeRole
      Policies:
        - PolicyName: MyLambdaPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: arn:aws:logs:*:*:*

  MyApiIntegration:
    Type: AWS::ApiGatewayV2::Integration
    Properties:
      PayloadFormatVersion: "2.0"
      ApiId: !Ref MyHttpApi
      IntegrationType: AWS_PROXY
      IntegrationUri: !Join
        - ''
        - - 'arn:'
          - !Ref 'AWS::Partition'
          - ':apigateway:'
          - !Ref 'AWS::Region'
          - ':lambda:path/2015-03-31/functions/'
          - !GetAtt MyLambdaFunction.Arn
          - /invocations
      IntegrationMethod: POST # Lambda invocations are typically POST
      # You'll also need to define mapping templates with AWS integration type
      # as shown in the IntegrationRequest and IntegrationResponse sections

  MyApiRoute:
    Type: AWS::ApiGatewayV2::Route
    Properties:
      ApiId: !Ref MyHttpApi
      RouteKey: GET /travel-plans # Changed to POST to match IntegrationMethod
      Target: !Join
        - /
        - - integrations
          - !Ref MyApiIntegration
      AuthorizationType: JWT
      AuthorizerId: !Ref MyJwtAuthorizer

1. Navegue até CloudFormation na região us-west-2 e clique em Create stack.
2. Selecione **With new resources(standard)**.
<figure>
    <img src="images/20.png">
</figure>

3. Para o pré-requisito, selecione **Choose an existing template**.
4. Para specify template, selecione **Upload a template file** e escolha o arquivo **template.yaml** salvo no passo anterior.
5. Clique em **Next**.

<figure>
    <img src="images/21.png">
</figure>

6. Forneça um **stack name** (ex: test-deployment-stack)
7. Insira o **Audience** para o parâmetro JwtAudienceList.
8. Insira o **Issuer URL** para o JwtIssueURL (ex: https://{yoursubdomain}.okta.com/oauth2/default).

<figure>
    <img src="images/22.png">
</figure>

9. Marque a caixa **I acknowledge that AWS CloudFormation might create IAM resources**.
10. Clique em **Next**.
<figure>
    <img src="images/32.png">
</figure>

11. Clique em **Submit**.

<figure>
    <img src="images/49.png">
</figure>

## Atualizando a Função Lambda

1. Navegue até **Lambda** > **Functions**, depois **clique** em **MyLambdaFunction**.
<figure>
    <img src="images/23.png">
</figure>

2. Copie o código-fonte abaixo na seção **Code**.
3. Renomeie o arquivo <b>index.js</b> para <b>lambda_function.py</b>
4. Clique em **Deploy**.
<figure>
    <img src="images/24.png">
</figure>

5. Navegue até **API Gateway** > **Integrations** e clique em **Manage integrations**.
<figure>
    <img src="images/56.png">
</figure>

6. Clique em **Edit** em **Integration details**.
<figure>
    <img src="images/57.png">
</figure>

7. Atualize para o ARN da **Lambda function** mais recente e clique em **Save**.
<figure>
    <img src="images/58.png">
</figure>

Código AWS Lambda:

In [ ]:
import json
from datetime import datetime, timedelta
import random
import uuid
# Mock data storage (in production, this would be a database)
MOCK_TRAVEL_PLANS = [
    {
        "id": "plan-001",
        "user_id": "user-123",
        "email": "john.doe@example.com",
        "destination": "Paris, France",
        "departure_date": "2024-03-15",
        "return_date": "2024-03-22",
        "accommodation": "Hotel Le Marais",
        "activities": ["Eiffel Tower", "Louvre Museum", "Seine River Cruise"],
        "budget": 2500.00,
        "status": "confirmed"
    },
    {
        "id": "plan-002",
        "user_id": "user-123",
        "email": "john.doe@example.com",
        "destination": "Tokyo, Japan",
        "departure_date": "2024-05-10",
        "return_date": "2024-05-20",
        "accommodation": "Tokyo Grand Hotel",
        "activities": ["Mount Fuji", "Sensoji Temple", "Shibuya Crossing"],
        "budget": 3500.00,
        "status": "planned"
    },
    {
        "id": "plan-003",
        "user_id": "user-456",
        "email": "jane.smith@example.com",
        "destination": "New York, USA",
        "departure_date": "2024-04-01",
        "return_date": "2024-04-07",
        "accommodation": "Manhattan Plaza Hotel",
        "activities": ["Statue of Liberty", "Central Park", "Broadway Show"],
        "budget": 2000.00,
        "status": "confirmed"
    },
    {
        "id": "plan-004",
        "user_id": "user-456",
        "email": "jane.smith@example.com",
        "destination": "Barcelona, Spain",
        "departure_date": "2024-06-15",
        "return_date": "2024-06-25",
        "accommodation": "Barcelona Beach Resort",
        "activities": ["Sagrada Familia", "Park Güell", "Las Ramblas"],
        "budget": 2800.00,
        "status": "planned"
    },
    {
        "id": "plan-005",
        "user_id": "user-789",
        "email": "bob.wilson@example.com",
        "destination": "Sydney, Australia",
        "departure_date": "2024-07-20",
        "return_date": "2024-08-03",
        "accommodation": "Sydney Harbour Hotel",
        "activities": ["Opera House", "Harbour Bridge", "Bondi Beach"],
        "budget": 4500.00,
        "status": "tentative"
    }
]
def lambda_handler(event, context):
    """
    Main Lambda handler for travel plans API
    """
    query_parameters = event.get('queryStringParameters', {})
    try:
        # Route based on HTTP method and path
        return get_travel_plans(query_parameters)
    except Exception as e:
        return create_response(500, {
            'error': 'Internal Server Error',
            'message': str(e)
        })
def create_response(status_code, body):
    """
    Create API Gateway Lambda response
    """
    return {
        'statusCode': status_code,
        'body': json.dumps(body)
    }
def get_travel_plans(query_params):
    """
    Get travel plans by user_id or email
    Query parameters:
    - user_id: Filter by user ID
    - email: Filter by email address
    """
    user_id = query_params.get('user_id') if query_params else None
    email = query_params.get('email') if query_params else None
    
    # Validate that at least one parameter is provided
    if not user_id and not email:
        return create_response(400, {
            'error': 'Either user_id or email must be provided',
            'message': 'Please provide user_id or email as query parameter'
        })
    
    # Filter travel plans based on the provided parameter
    filtered_plans = []
    
    for plan in MOCK_TRAVEL_PLANS:
        if user_id and plan['user_id'] == user_id:
            filtered_plans.append(plan)
        elif email and plan['email'].lower() == email.lower():
            filtered_plans.append(plan)
    
    # Sort by departure date (most recent first)
    filtered_plans.sort(key=lambda x: x.get('departure_date', ''), reverse=True)
    
    # Return response
    if filtered_plans:
        return create_response(200, {
            'success': True,
            'count': len(filtered_plans),
            'travel_plans': filtered_plans,
            'filter': {
                'user_id': user_id,
                'email': email
            }
        })
    else:
        return create_response(404, {
            'success': False,
            'message': 'No travel plans found for the specified user',
            'filter': {
                'user_id': user_id,
                'email': email
            },
            'travel_plans': []
        })


## Criando o Amazon Bedrock AgentCore Gateway

O código abaixo irá escrever um arquivo **demo_openapi.yaml** no seu disco rígido. Este arquivo será usado para criar o Amazon Bedrock AgentCore Gateway que fará proxy de requisições para o API Gateway que criamos nos passos anteriores.
<ol type="1">
<li>Navegue até <b>API Gateway</b> > <b>APIs</b> > <b>Stages</b> e clique em <b>Create</b>.</li>
<figure>
    <img src="images/50.png">
</figure>
<li>Defina o <b>Name</b> do stage como <b>default</b>.</li>
<li>Clique para <b>Enable automatic deployment</b>.</li>
<li>Clique em <b>Create</b>.</li>
<figure>
    <img src="images/51.png">
</figure>
<li>Anote o <b>Default endpoint</b>.</li>
<figure>
    <img src="images/52.png">
</figure>
<li>Clique na célula de código abaixo para salvar o arquivo <b>demo_openapi.yaml</b> no seu disco rígido.</li>
    <ol type="a">
            <li>Abra <b>demo_openapi.yaml</b> e edite o <b>{yoursubdomain}</b> da <b>server URL</b> para corresponder ao <b>Invoke URL</b> do último passo.</li>
            <li>Edite o <b>{yoursubdomain}</b> da <b>authorization URL</b> e do <b>token URL</b> para corresponder ao seu tenant Okta, depois salve o arquivo (localizado na URL usada para fazer login no seu tenant Okta).</li>
    </ol>
</ol>

In [ ]:
%%writefile demo_openapi.yaml
openapi: 3.0.0
info:
  title: Travel Plans API
  description: API for retrieving user travel plans with secure authentication
  version: 1.0.0
  contact:
    name: Travel Plans Developer
    email: developer@example.com
  license:
    name: Apache 2.0
    url: http://www.apache.org/licenses/LICENSE-2.0.html

servers:
  - url: https://{yoursubdomain}.execute-api.us-west-2.amazonaws.com/default
    description: Production server

paths:
  /travel-plans:
    get:
      summary: Retrieve Travel Plans
      description: Fetch travel plans by user ID or email
      operationId: getTravelPlans
      security:
        - OAuth2:
          - read:travel-plans
        - ApiKeyAuth: []
      parameters:
        - in: query
          name: user_id
          schema:
            type: string
          required: false
          description: Unique identifier of the user
          example: "user-123"
        
        - in: query
          name: email
          schema:
            type: string
            format: email
          required: false
          description: Email address of the user
          example: "john.doe@example.com"
      
      responses:
        '200':
          description: Successful retrieval of travel plans
          content:
            application/json:
              schema:
                type: object
                properties:
                  success:
                    type: boolean
                  count:
                    type: integer
                  travel_plans:
                    type: array
                    items:
                      $ref: '#/components/schemas/TravelPlan'
                  filter:
                    type: object
                    properties:
                      user_id:
                        type: string
                      email:
                        type: string
        
        '400':
          description: Bad Request - Missing query parameters
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ErrorResponse'
        
        '401':
          description: Unauthorized - Invalid or missing authentication
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ErrorResponse'
        
        '403':
          description: Forbidden - Insufficient permissions
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ErrorResponse'
        
        '404':
          description: No travel plans found
          content:
            application/json:
              schema:
                type: object
                properties:
                  success:
                    type: boolean
                  message:
                    type: string
                  filter:
                    type: object
                    properties:
                      user_id:
                        type: string
                      email:
                        type: string
                  travel_plans:
                    type: array
                    items: {}
        
        '500':
          description: Internal Server Error
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ErrorResponse'

components:
  securitySchemes:
    OAuth2:
      type: oauth2
      flows:
        authorizationCode:
          authorizationUrl: https://{yoursubdomain}.okta.com/oauth2/default/v1/authorize
          tokenUrl: https://{yoursubdomain}.okta.com/oauth2/default/v1/token
          scopes:
            read:travel-plans: Read access to travel plans
            write:travel-plans: Write access to travel plans
            delete:travel-plans: Delete access to travel plans
  
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key

    BearerAuth:
      type: http
      scheme: bearer
      bearerFormat: JWT

  schemas:
    TravelPlan:
      type: object
      required:
        - id
        - user_id
        - email
        - destination
        - departure_date
        - return_date
        - status
      properties:
        id:
          type: string
          description: Unique identifier for the travel plan
          example: "plan-001"
        user_id:
          type: string
          description: Unique identifier of the user
          example: "user-123"
        email:
          type: string
          format: email
          description: Users email address
          example: "john.doe@example.com"
        destination:
          type: string
          description: Travel destination
          example: "Paris, France"
        departure_date:
          type: string
          format: date
          description: Date of departure
          example: "2024-03-15"
        return_date:
          type: string
          format: date
          description: Date of return
          example: "2024-03-22"
        accommodation:
          type: string
          description: Name of accommodation
          example: "Hotel Le Marais"
        activities:
          type: array
          items:
            type: string
          description: List of planned activities
          example: ["Eiffel Tower", "Louvre Museum"]
        budget:
          type: number
          format: float
          description: Estimated travel budget
          example: 2500.00
        status:
          type: string
          description: Status of the travel plan
          enum:
            - confirmed
            - planned
            - tentative
          example: "confirmed"

    ErrorResponse:
      type: object
      properties:
        error:
          type: string
        message:
          type: string
        error_code:
          type: string
        timestamp:
          type: string
          format: date-time

    OAuthToken:
      type: object
      properties:
        access_token:
          type: string
          description: JWT access token
        token_type:
          type: string
          enum:
            - Bearer
        expires_in:
          type: integer
          description: Token expiration time in seconds
        refresh_token:
          type: string
          description: Token to obtain a new access token

tags:
  - name: Travel Plans
    description: Operations related to travel plan retrieval
  - name: Authentication
    description: OAuth2 and API Key authentication methods

x-security-definitions:
  - name: OAuth2
    description: >
      OAuth 2.0 Authentication:
      - Authorization Code Flow
  - name: API Key
    description: >
      API Key authentication for service-to-service communication

x-rate-limiting:
  limit: 100
  period: 1 minute
  
x-error-handling:
  generic-errors:
    - 400: Bad Request
    - 401: Unauthorized
    - 403: Forbidden
    - 404: Not Found
    - 500: Internal Server Error

2. Faça upload de **demo_openapi.yaml** para um bucket S3. Você pode escolher para qual bucket fazer o upload.
<figure>
    <img src="images/25.png">
</figure>

3. Navegue até **Amazon Bedrock AgentCore**.
4. No menu do lado esquerdo, selecione **Identity**
5. Clique em **Add OAuth client / API key**, depois **Add Oauth client**.
<figure>
    <img src="images/26.png">
</figure>

6. Em Provider, selecione **Custom provider**.
7. Selecione **Discovery URL** e insira o **Client ID**, **Client secret** e a **Discovery URL** (https://{yoursubdomain}.okta.com/oauth2/default/.well-known/openid-configuration), depois salve o nome do provedor de recursos para uso futuro.
8. Clique em **Add OAuth Client**.
<figure>
    <img src="images/27.png">
</figure>


9. No menu do lado esquerdo, selecione **Gateways**, depois clique em **Create Gateway**.
<figure>
    <img src="images/28.png">
</figure>

10. Defina um **Gateway name** ou use o nome padrão.
<figure>
    <img src="images/29.png">
</figure>

11. Selecione **Use existing Identity provider configurations** em **Inbound Auth configurations**.
12. Insira a **Discovery URL** (https://{yoursubdomain}.okta.com/oauth2/default/.well-known/openid-configuration), **Allowed audiences** e **Allowed clients** (audience e Client ID para seu tenant Okta).
<figure>
    <img src="images/30.png">
</figure>

13. Em Target:{target name}, selecione **REST API** para target type.
14. Para REST API type, selecione **OpenAPI schema**.
15. Para OpenAPI schema, selecione **Define with an S3 resource**.
16. Navegue até a localização do documento openapi no **passo 2** e selecione-o.
17. Para as Outbound Auth configurations, selecione **OAuth client**.
18. Em OAuth client, selecione o Amazon Bedrock AgentCore Identity criado nos **passos 3 a 5**.
<figure>
    <img src="images/31.png">
</figure>

19. Em Scopes, adicione <b>okta.myAccount.read</b>.
20. Clique em **Save**.
<figure>
    <img src="images/33.png">
</figure>



## Preparando o agente para implantação no AgentCore Runtime

Este código define um chatbot **Travel Assistant** usando o framework Strands e Bedrock AgentCore SDK.

Ele configura um agente assistente de viagem que pode buscar planos de viagem existentes usando ID de cliente ou email via um Amazon Bedrock AgentCore Gateway. O decorador @requires_access_token() manipula autenticação OAuth 2.0 gerenciando automaticamente o processo de obtenção e injeção de um token de acesso na função decorada. Ele configura o fluxo de autenticação com Okta usando o cliente OAuth de saída que criamos nos passos anteriores, solicita os scopes necessários ("okta.myAccount.read"), e quando autenticação é necessária, imprime a URL de autorização no console para o usuário visitar e autorizar a aplicação. Uma vez que o usuário complete o fluxo OAuth, o decorador injeta automaticamente o token de acesso obtido como parâmetro na função need_token_3LO_async, e armazena o token no cofre de tokens para evitar requisições de autenticação repetidas.

1. Execute a célula abaixo para salvar o código do agente como **my_agent_mcp.py** no seu sistema de arquivos local.
2. Navegue até <b>Amazon AgentCore Gateway</b> > <b>Gateways</b> e clique no seu gateway (ex: gateway-quick-start-234a1).
<figure>
    <img src="images/53.png">
</figure>

3. Localize o arquivo e substitua **{yoursubdomain}** para a gateway_url para corresponder à **Gateway resource URL**.
<figure>
    <img src="images/38.png">
</figure>

4. Para provider_name, substitua **{yourprovidername}** pelo nome do seu cliente OAuth de saída (ex: resource-provider-oauth-client-1wbak).
5. Salve o arquivo.

In [ ]:
%%writefile my_agent_mcp.py
import json
import requests
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent, tool
from strands.tools.mcp import MCPClient
from strands_tools import calculator, current_time

# Import the AgentCore SDK
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.identity.auth import requires_access_token

WELCOME_MESSAGE = """
Welcome to the Travel Assistant! How can I help you today?
"""

SYSTEM_PROMPT = """
You are an helpful travel support assistant.
When provided with a customer email, gather all necessary info and prepare the response.
When asked about existing travel plans, look for it and customize the summary based on the prompt.
Don't mention the customer ID in your reply.
"""

# Global token storage
okta_access_token = None

def create_streamable_http_transport(mcp_url: str, access_token: str):
       return streamablehttp_client(mcp_url, headers={"Authorization": f"Bearer {access_token}"})

# Create an AgentCore app
app = BedrockAgentCoreApp()

async def agent_task(user_message: str) -> None:

    global okta_access_token
    okta_access_token = await need_token_3LO_async(access_token='')
    response = ''

    gateway_url = "https://{yoursubdomain}.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp"
    mcp_client = MCPClient(lambda: create_streamable_http_transport(gateway_url , okta_access_token))

    with mcp_client:
        #tools = await get_full_tools_list(mcp_client)
        #print(f"Found the following tools: {[tool.tool_name for tool in tools]}")
    
        agent = Agent(tools=mcp_client.list_tools_sync())
        response = agent(user_message)
        print(response)
    
    return response.message['content'][0]['text']

# Injects Okta Access Token
@requires_access_token(# Uses the same credential provider name created above
    provider_name= "{yourprovidername}",
    # Requires Okta OAuth2 scope to access MCP Server
    scopes= ["okta.myAccount.read"],
    # Sets to OAuth 2.0 Authorization Code flow
    auth_flow= "USER_FEDERATION",
    # Prints authorization URL to console
    on_auth_url= lambda x: print("\nPlease copy and paste this URL in your browser:\n" + x),
    # If false, caches obtained access token
    force_authentication= False,) 
async def need_token_3LO_async(*, access_token: str) -> str:
    """Handle OAuth authentication flow."""
    global okta_access_token
    okta_access_token = access_token
    return access_token


# Specify the entry point function invoking the agent
@app.entrypoint
async def invoke(payload):
    """Handler for agent invocation"""
    user_message = payload.get(
        "prompt", "No prompt found in input, please guide customer to create a json payload with prompt key"
    )

    result = await agent_task(user_message)
    return result

if __name__ == "__main__":
    app.run()

Para este código funcionar, os módulos Strands Agents precisam estar instalados no ambiente Python.

Para instalar dependências, crie e ative um ambiente virtual:

In [ ]:
!python -m venv .venv 
!source .venv/bin/activate

Adicione os módulos Strands Agents, AgentCore SDK e AgentCore starter toolkit ao arquivo de dependências e salve-o como **requirements.txt**:

In [ ]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore
bedrock-agentcore-starter-toolkit


Depois instale todos os requisitos no ambiente virtual:

In [ ]:
%pip install -r requirements.txt

## Implantando o agente no AgentCore Runtime
A operação `CreateAgentRuntime` suporta opções de configuração abrangentes, permitindo que você especifique imagens de container, variáveis de ambiente e configurações de criptografia. Você também pode configurar definições de protocolo (HTTP, MCP) e mecanismos de autorização para controlar como seus clientes se comunicam com o agente.

Nota: A melhor prática operacional é empacotar código como container e fazer push para ECR usando pipelines CI/CD e IaC

Neste tutorial usaremos o Amazon Bedrock AgentCode Python SDK para facilmente empacotar seus artefatos e implantá-los no agentcore runtime.

### Configurar agente para implantação no AgentCore Runtime
Em seguida usaremos nosso starter toolkit para configurar a implantação do AgentCore Runtime com um entrypoint, a role de execução que acabamos de criar e um arquivo requirements. Também configuraremos o starter kit para criar automaticamente o repositório Amazon ECR no lançamento.

Durante o passo de configuração, seu arquivo docker será gerado com base no código da sua aplicação

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

discovery_url = input("Enter your discovery URL: ")

client_id = input("Enter your client ID: ")

audience = input("Enter your audience: ")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="my_agent_mcp.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_agent_inbound_identity_okta",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
            "allowedAudience": [audience]
        }
    }
)
response

### Revisar a configuração do AgentCore

In [ ]:
!cat .bedrock_agentcore.yaml

#### Lançando agente no AgentCore Runtime

Agora que temos um arquivo docker, vamos lançar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime.

<figure>
    <img src="images/14.png">
</figure>

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

#### Verificando o Status do AgentCore Runtime

Agora que implantamos o AgentCore Runtime, vamos verificar seu status de implantação.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Dar à Role de Execução Permissão para Ler Token OAuth

1. Navegue até <b>Amazon Bedrock AgentCore</b>.
2. Clique em <b>Agent Runtime</b> no menu do lado esquerdo.
3. Clique no seu agente (ex: strands_agent_inbound_identity_okta).
<figure>
    <img src="images/45.png">
</figure>

4. Clique na versão mais recente.
<figure>
    <img src="images/46.png">
</figure>

5. Clique no <b>IAM service role</b>.
<figure>
    <img src="images/47.png">
</figure>

6. Clique no <b>Policy name</b> em <b>Permissions policies</b>.
<figure>
    <img src="images/48.png">
</figure>

7. Adicione a seguinte declaração para dar ao seu agente acesso ao cofre de tokens.

		{
			"Sid": "GetResourceOauth2Token",
			"Effect": "Allow",
			"Action": [
				"bedrock-agentcore:GetResourceOauth2Token",
				"secretsmanager:GetSecretValue"
			],
			"Resource": "*"
		}

8. Clique em <b>Next</b>.
<figure>
    <img src="images/59.png">
</figure>

### Conceda ao seu Agente Acesso ao Modelo Amazon Bedrock
1. Navegue até <b>Amazon Bedrock</b> > <b>Model access</b> e clique em <b>Modify model access</b>.
<figure>
    <img src="images/54.png">
</figure>

2. Marque <b>Claude Sonnet 4</b> e clique em <b>Next</b> (Nota: o modelo padrão usado por strands pode mudar).
<figure>
    <img src="images/55.png">
</figure>

#### Invocando AgentCore Runtime sem

Finalmente, podemos invocar nosso AgentCore Runtime com um payload. Tente executar a célula seguinte e você verá um erro que diz **"AccessDeniedException: An error occurred (AccessDeniedException) when calling the InvokeAgentRuntime operation: Agent is configured for a different authorization token type".**

<figure>
    <img src="images/15.png">
</figure>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What are the travel plans for customer 123?"})
invoke_response

#### Invocando AgentCore Runtime com autorização

Vamos invocar o agente com o tipo de token de autorização correto. No nosso caso, será o token de acesso Okta.

O comando seguinte instala a biblioteca **Requests** do framework **Flask** necessária para o servidor OAuth no próximo passo.

In [ ]:
%pip install flask requests

O código seguinte irá levantar um servidor para que você consiga recuperar o token de acesso. Uma vez iniciado, clique no link ou navegue até http://127.0.0.1:5000 para completar o fluxo OAuth. Depois de recuperar o token de acesso, pare o servidor da mesma forma que foi iniciado.

O código seguinte:
1. Cria um pequeno servidor web com Flask para manipular autenticação OAuth.
2. Quando o usuário visita ```/login```, eles são redirecionados para o Okta para fazer login.
3. Após o login, o servidor recebe um código de autorização via a rota /callback.
4. O código é então trocado por um token de acesso via uma requisição POST para o endpoint de token.
5. Se bem-sucedido, o token de acesso é salvo na sessão e impresso no console.

**Nota**: Você pode encontrar as URLs de autorização e token da sua URL de descoberta (https://{yoursubdomain}.okta.com/oauth2/default/.well-known/openid-configuration).

In [ ]:
import os
import requests
import secrets
from flask import Flask, redirect, request, session, url_for

app = Flask(__name__)
app.secret_key = os.urandom(24)

# === Configuration ===
CLIENT_ID = input("Enter your Client ID: ")
CLIENT_SECRET = input("Enter your Client Secret: ")
AUTHORIZATION_BASE_URL = input("Enter your authorization URL: ")
TOKEN_URL = input("Enter your token URL: ")
REDIRECT_URI = "http://127.0.0.1:5000/callback"
SCOPE = "openid email"  # Adjust according to the provider

# === Step 1: Redirect to Authorization Server ===
@app.route("/")
def home():
    return '<a href="/login">Login with OAuth</a>'

@app.route("/login")
def login():
    state = secrets.token_urlsafe(16)
    session["oauth_state"] = state

    auth_url = (
        f"{AUTHORIZATION_BASE_URL}?response_type=code&client_id={CLIENT_ID}"
        f"&redirect_uri={REDIRECT_URI}&scope={SCOPE}&state={state}"
    )
    return redirect(auth_url)

# === Step 2: Handle Callback and Exchange Code for Token ===
@app.route("/callback")
def callback():
    error = request.args.get("error")
    if error:
        return f"Error: {error}"

    code = request.args.get("code")
    if not code:
        return "No code found"

    token_data = {
        "grant_type": "authorization_code",
        "code": code,
        "redirect_uri": REDIRECT_URI,
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    }

    token_response = requests.post(TOKEN_URL, data=token_data)
    token_json = token_response.json()

    if "access_token" in token_json:
        session["access_token"] = token_json["access_token"]
        print("Access Token: " + token_json["access_token"])

        return "Login successful! Access token in logs."
    else:
        return f"Failed to get token: {token_json}"

if __name__ == "__main__":
    app.run(host='127.0.0.1', port=5000)

Copie o token de acesso e insira-o quando solicitado na próxima seção de código.

In [ ]:
bearer_token = input("Enter your bearer token: ")
invoke_response = agentcore_runtime.invoke(
    {"prompt": "What flights does customer with user id user-123 have scheduled?"}, 
    bearer_token=bearer_token
)
invoke_response

1. No Console AWS, navegue até Amazon Bedrock AgentCore e clique em Agent Runtime em Build and Deploy
2. Clique no seu agente (ex: my_agent_mcp).
<figure>
    <img src="images/36.png">
</figure>

3. Copie o **Runtime ID**.
<figure>
    <img src="images/34.png">
</figure>

4. Execute o seguinte comando em um terminal para recuperar logs do agente. Substitua **{myagentruntimeid}** pelo **Runtime ID** do seu agente.

```aws logs tail /aws/bedrock-agentcore/runtimes/{myagentruntimeid} --follow```

5. Capture a URL de requisição de autorização e cole-a em um navegador para completar o handshake OAuth.

<figure>
    <img src="images/35.png">
</figure>

## Parabéns!